In [2]:
# ============================================================================
# 02_run_generation.ipynb
# ----------------------------------------------------------------------------
# Beyond Full-Schema Prompting: A Graph-based Semantic Layer for Text-to-SQL
#
# Core experiment loop. For each (dataset, condition, model, question):
#   1. Load the pre-built schema context (notebook 01).
#   2. Prompt the LLM to generate SQL.
#   3. Validate with SQLGlot (Invalid SQL rate).
#   4. Execute predicted vs gold SQL on the SQLite DB; compare result sets (EX).
#   5. Compute schema-linking recall/precision vs gold tables/columns.
#   6. Record tokens, latency, estimated cost.
# Writes one tidy master table consumed by all RQ notebooks (03-08).
#
# v4 progress reporting: shows overall progress across all cells, per-cell cache
# hit/miss counts, live API-call rate and mean latency, so slowdowns are visible
# (a slow patch = real API calls; a fast patch = cache hits). SMOKE_TEST switch
# for a tiny cheap run. Anonymity: no filesystem paths are printed.
# ============================================================================


# %% [markdown]
# ## Cell 1 - Load shared configuration

# %%
%run 00_setup_and_config.ipynb


# %%
import re
import time
import sqlglot
from tqdm.auto import tqdm

print("Notebook 02 dependencies ready.")


# %% [markdown]
# ## Cell 2 - Run controls (smoke switch) + experiment plan

# %%
SMOKE_TEST = False                # flip True for a tiny cheap run
SMOKE_LIMIT = 5
FULL_LIMIT = None

GEN_MAX_TOKENS = 512

SMOKE_PLAN = [
    ("D1", ["C1", "C4"], ["STRONG"]),
]

FULL_PLAN = [
    ("D1", ["C1", "C2", "C3", "C4", "C5_L1", "C5_L2", "C5_L3"],
     ["STRONG", "MID", "WEAK"]),
    ("D2", ["C1", "C2", "C3", "C4", "C5_L1", "C5_L2", "C5_L3"],
     ["STRONG", "MID", "WEAK"]),
    ("D2_fin", ["C1", "C4", "C5_L1", "C5_L2"],
     ["STRONG", "MID", "WEAK"]),
]

if SMOKE_TEST:
    EXPERIMENT_PLAN = SMOKE_PLAN
    RUN_LIMIT = SMOKE_LIMIT
else:
    EXPERIMENT_PLAN = FULL_PLAN
    RUN_LIMIT = FULL_LIMIT

print(f"Mode: {'SMOKE' if SMOKE_TEST else 'FULL'} | "
      f"cells={sum(len(cs)*len(ts) for _, cs, ts in EXPERIMENT_PLAN)} | "
      f"limit={RUN_LIMIT}")


# %% [markdown]
# ## Cell 3 - Prompt template + output cleaning

# %%
SQL_SYSTEM = (
    "You are an expert data analyst that writes correct, executable SQLite SQL. "
    "Use ONLY the tables and columns provided in the schema context. "
    "Return a single SQL query and nothing else - no explanation, no markdown."
)

SQL_PROMPT_TEMPLATE = """{schema_context}

-- Question:
-- {question}

-- Write one SQLite query that answers the question.
-- Output only the SQL statement.
SQL:"""


def build_prompt(schema_context, question):
    return SQL_PROMPT_TEMPLATE.format(schema_context=schema_context, question=question)


def clean_sql(text):
    """Strip markdown fences / stray prose the model may add."""
    t = text.strip()
    t = re.sub(r"^```[a-zA-Z]*", "", t).strip()
    t = re.sub(r"```$", "", t).strip()
    m = re.search(r"(?is)\b(WITH|SELECT)\b", t)
    if m:
        t = t[m.start():]
    if ";" in t:
        t = t[: t.index(";") + 1]
    return t.strip()


# %% [markdown]
# ## Cell 4 - SQLGlot validation (Invalid SQL rate)

# %%
def validate_sql(sql, dialect="sqlite"):
    try:
        return (sqlglot.parse_one(sql, read=dialect) is not None), ""
    except Exception as e:
        return False, str(e)


# %% [markdown]
# ## Cell 5 - Execution-based comparison (Execution Accuracy)
# sqlite3's connect(timeout=) governs LOCK waits, not query runtime. A
# pathological SQL (cartesian blow-up) can hang forever, so we enforce a real
# wall-clock limit via set_progress_handler, which fires every N bytecode ops and
# aborts once the deadline passes.

# %%
def _db_path(dataset_key, db_id):
    return DATA_DIR / dataset_key / "db" / f"{db_id}.sqlite"


def run_sql(dataset_key, db_id, sql, wall_limit_s=10.0):
    """
    Execute SQL with a real wall-clock limit. Returns (rows, error).
    rows is a list of tuples, or None on error/timeout.
    """
    path = _db_path(dataset_key, db_id)
    if not path.exists():
        return None, "DB not found"

    con = None
    deadline = time.time() + wall_limit_s

    def _guard():
        # Called periodically by SQLite; returning non-zero aborts the query.
        return 1 if time.time() > deadline else 0

    try:
        con = sqlite3.connect(str(path), timeout=5.0)
        con.text_factory = lambda b: b.decode("utf-8", "ignore")
        con.set_progress_handler(_guard, 1_000_000)
        cur = con.cursor()
        cur.execute(sql)
        rows = cur.fetchall()
        con.close()
        return rows, ""
    except Exception as e:
        if con is not None:
            try:
                con.close()
            except Exception:
                pass
        msg = str(e)
        if "interrupted" in msg.lower() or "aborted" in msg.lower():
            return None, f"SQL wall-clock timeout (> {wall_limit_s:.0f}s)"
        return None, msg


def _has_order_by(sql):
    return bool(re.search(r"(?is)\border\s+by\b", sql))


def compare_results(pred_rows, gold_rows, ordered):
    if pred_rows is None or gold_rows is None:
        return False
    def norm(rows):
        return [tuple("" if v is None else str(v) for v in r) for r in rows]
    p, g = norm(pred_rows), norm(gold_rows)
    if ordered:
        return p == g
    from collections import Counter
    return Counter(p) == Counter(g)


def execution_match(dataset_key, db_id, pred_sql, gold_sql):
    gold_rows, gold_err = run_sql(dataset_key, db_id, gold_sql)
    pred_rows, pred_err = run_sql(dataset_key, db_id, pred_sql)
    if gold_rows is None:
        return None, pred_err, gold_err     # unscoreable (e.g. DB missing)
    return compare_results(pred_rows, gold_rows, _has_order_by(gold_sql)), pred_err, gold_err


# %% [markdown]
# ## Cell 6 - Schema-linking metrics (strict + relaxed)

# %%
def _norm(names):
    return {n.strip().lower() for n in names if n and n.strip()}

def _suffix(name):
    return name.split(".")[-1]

def link_scores_strict(selected, gold):
    g, s = _norm(gold), _norm(selected)
    if not g:
        return np.nan, np.nan
    tp = len(g & s)
    return tp / len(g), (tp / len(s) if s else 0.0)

def link_scores_relaxed(selected, gold):
    g = {_suffix(x) for x in _norm(gold)}
    s = {_suffix(x) for x in _norm(selected)}
    if not g:
        return np.nan, np.nan
    tp = len(g & s)
    return tp / len(g), (tp / len(s) if s else 0.0)

def table_scores(selected, gold):
    g, s = _norm(gold), _norm(selected)
    if not g:
        return np.nan, np.nan
    tp = len(g & s)
    return tp / len(g), (tp / len(s) if s else 0.0)


# %% [markdown]
# ## Cell 7 - Lookups for gold questions + context metadata

# %%
def load_questions(dataset_key):
    data = json.loads((DATA_DIR / dataset_key / "questions.json").read_text(encoding="utf-8"))
    return {q["qid"]: q for q in data}

def load_context_meta(dataset_key):
    path = CONTEXTS_DIR / f"{dataset_key}__context_meta.csv"
    if not path.exists():
        return None
    df = pd.read_csv(path)
    return {(r.qid, r.condition): r for r in df.itertuples()}


# %% [markdown]
# ## Cell 8 - Plan sizing (for the global progress bar)
# Pre-compute how many questions each cell has so we can show overall progress
# (cell k / K, and question n / N across the whole run).

# %%
def plan_sizes(plan, limit):
    """Return (list of (ds,cond,tier,n_questions), total_questions)."""
    cells, total = [], 0
    for ds, conds, tiers in plan:
        meta = load_context_meta(ds)
        if meta is None:
            # contexts not built; count 0 so the cell is skipped gracefully
            for cond in conds:
                for tier in tiers:
                    cells.append((ds, cond, tier, 0))
            continue
        for cond in conds:
            n = len({qid for (qid, c) in meta.keys() if c == cond})
            if limit:
                n = min(n, limit)
            for tier in tiers:
                cells.append((ds, cond, tier, n))
                total += n
    return cells, total

PLAN_CELLS, PLAN_TOTAL = plan_sizes(EXPERIMENT_PLAN, RUN_LIMIT)
print(f"Planned: {len(PLAN_CELLS)} cells, {PLAN_TOTAL} question-generations total.")


# %% [markdown]
# ## Cell 9 - Single-cell runner with cache/API-aware progress
# The progress bar postfix shows: cache-hit count, real API-call count, and the
# rolling mean latency of REAL calls only. A slow stretch with rising 'api' and
# latency means real calls; a fast stretch with rising 'cache' means cache hits.

# %%
def run_cell(dataset_key, condition, model_tier, limit=None,
             global_bar=None):
    model_name = MODEL_TIERS[model_tier]
    questions = load_questions(dataset_key)
    meta = load_context_meta(dataset_key)
    if meta is None:
        print(f"  [skip] {dataset_key}: contexts not built (run notebook 01).")
        return pd.DataFrame()

    qids = sorted({qid for (qid, cond) in meta.keys() if cond == condition})
    if limit:
        qids = qids[:limit]

    records = []
    n_cache, n_api, api_latency_sum = 0, 0, 0.0
    t_cell0 = time.time()

    bar = tqdm(qids, desc=f"{dataset_key}/{condition}/{model_tier}",
               leave=False, dynamic_ncols=True)
    for qid in bar:
        q = questions[qid]
        m = meta[(qid, condition)]
        schema_context = (PROJECT_ROOT / m.context_path).read_text(encoding="utf-8")

        resp = llm_complete(build_prompt(schema_context, q["question"]),
                            model=model_name, system=SQL_SYSTEM,
                            temperature=0.0, max_tokens=GEN_MAX_TOKENS)

        # cache/api accounting
        if resp.get("cached"):
            n_cache += 1
        else:
            n_api += 1
            api_latency_sum += resp.get("latency_s", 0.0) or 0.0
        mean_api_lat = (api_latency_sum / n_api) if n_api else 0.0
        bar.set_postfix(cache=n_cache, api=n_api,
                        api_lat=f"{mean_api_lat:.2f}s")
        if global_bar is not None:
            global_bar.update(1)

        pred_sql = clean_sql(resp["text"])
        is_valid, verr = validate_sql(pred_sql)
        if is_valid:
            match, pred_err, _ = execution_match(
                dataset_key, q["db_id"], pred_sql, q["gold_sql"])
        else:
            match, pred_err = False, verr

        sel_tables = str(m.selected_tables).split("|") if m.selected_tables else []
        sel_cols = str(m.selected_columns).split("|") if m.selected_columns else []
        t_recall, t_prec = table_scores(sel_tables, q.get("gold_tables", []))
        cs_recall, cs_prec = link_scores_strict(sel_cols, q.get("gold_columns", []))
        cr_recall, cr_prec = link_scores_relaxed(sel_cols, q.get("gold_columns", []))

        records.append({
            "dataset": dataset_key, "condition": condition,
            "model_tier": model_tier, "model_name": model_name,
            "qid": qid, "db_id": q["db_id"],
            "n_gold_tables": len(q.get("gold_tables", [])),
            "n_gold_cols": len(q.get("gold_columns", [])),
            "n_join_gold": max(0, len(q.get("gold_tables", [])) - 1),
            "exec_match": (np.nan if match is None else int(bool(match))),
            "valid_sql": int(bool(is_valid)),
            "table_recall": t_recall, "table_precision": t_prec,
            "col_recall_strict": cs_recall, "col_precision_strict": cs_prec,
            "col_recall_relaxed": cr_recall, "col_precision_relaxed": cr_prec,
            "context_tokens_est": m.context_tokens_est,
            "prompt_tokens": resp["prompt_tokens"],
            "completion_tokens": resp["completion_tokens"],
            "total_tokens": resp["total_tokens"],
            "cost_usd": estimate_cost(model_name, resp["prompt_tokens"],
                                      resp["completion_tokens"]),
            "latency_s": resp["latency_s"],
            "cached": int(resp["cached"]),
            "pred_sql": pred_sql, "pred_error": pred_err,
        })

    bar.close()
    df = pd.DataFrame(records)
    elapsed = time.time() - t_cell0
    ex = df["exec_match"].mean(skipna=True) if len(df) else np.nan
    ex_str = "n/a" if (len(df) == 0 or np.isnan(ex)) else f"{ex:.3f}"
    print(f"  {dataset_key}/{condition}/{model_tier}: "
          f"EX={ex_str} valid={df['valid_sql'].mean():.3f} n={len(df)} "
          f"| cache={n_cache} api={n_api} "
          f"| {elapsed:.0f}s")
    return df


# %% [markdown]
# ## Cell 10 - Execute the plan (with a global progress bar)
# The outer bar tracks total question-generations across ALL cells so you can see
# overall speed and ETA; each cell also prints its own cache/api breakdown.

# %%
all_results = []
done_cells = 0
global_bar = tqdm(total=PLAN_TOTAL, desc="OVERALL", position=0,
                  dynamic_ncols=True)

for dataset_key, conditions, tiers in EXPERIMENT_PLAN:
    if not (DATA_DIR / dataset_key / "questions.json").exists():
        print(f"[skip] {dataset_key}: questions.json missing.")
        continue
    for cond in conditions:
        for tier in tiers:
            done_cells += 1
            n_this = next((n for (d, c, t, n) in PLAN_CELLS
                           if d == dataset_key and c == cond and t == tier), 0)
            global_bar.set_postfix(cell=f"{done_cells}/{len(PLAN_CELLS)}",
                                   now=f"{dataset_key}/{cond}/{tier}")
            df_cell = run_cell(dataset_key, cond, tier, limit=RUN_LIMIT,
                               global_bar=global_bar)
            if len(df_cell):
                all_results.append(df_cell)

global_bar.close()
results = pd.concat(all_results, ignore_index=True) if all_results else pd.DataFrame()
if not len(results):
    print("No results produced. Check notebook 01 outputs.")


# %% [markdown]
# ## Cell 11 - Persist the master results table
# Smoke mode writes a separate file so it never overwrites a full run.

# %%
if len(results):
    out_name = "generation_results_smoke" if SMOKE_TEST else "generation_results"
    save_table(results, out_name)

    summary = (results.groupby(["dataset", "condition", "model_tier"])
               .agg(n=("qid", "count"),
                    EX=("exec_match", "mean"),
                    valid=("valid_sql", "mean"),
                    tbl_recall=("table_recall", "mean"),
                    tbl_prec=("table_precision", "mean"),
                    col_rec_strict=("col_recall_strict", "mean"),
                    col_prec_strict=("col_precision_strict", "mean"),
                    col_rec_relax=("col_recall_relaxed", "mean"),
                    col_prec_relax=("col_precision_relaxed", "mean"),
                    ctx_tokens=("context_tokens_est", "mean"),
                    cost=("cost_usd", "mean"))
               .round(4).reset_index())
    display(summary)
    save_table(summary, out_name + "_summary")

Core libraries imported.
Project directories ready.
Environment loaded. Model tiers:
  WEAK   -> gpt-4o-mini
  MID    -> gpt-5.4-mini
  STRONG -> gpt-5.4
Conditions: ['C1', 'C2', 'C3', 'C4', 'C5']
Datasets   : ['D1', 'D2', 'D2_fin', 'D3']
Grayscale plotting style configured.
Condition styles: {'C1': '0.15', 'C2': '0.3', 'C3': '0.45', 'C4': '0.6', 'C5': '0.75'}
Live check skipped (set RUN_LIVE_CHECK=True to test credentials).
Setup complete. Config snapshot saved.
Notebook 02 dependencies ready.
Mode: FULL | cells=54 | limit=None
Planned: 54 cells, 32598 question-generations total.


OVERALL:   0%|          | 0/32598 [00:00<?, ?it/s]

D1/C1/STRONG:   0%|          | 0/1034 [00:00<?, ?it/s]

  D1/C1/STRONG: EX=0.779 valid=0.999 n=1034 | cache=1034 api=0 | 17s


D1/C1/MID:   0%|          | 0/1034 [00:00<?, ?it/s]

...

  D2_fin/C5_L2/MID: EX=0.062 valid=1.000 n=32 | cache=32 api=0 | 4s


D2_fin/C5_L2/WEAK:   0%|          | 0/32 [00:00<?, ?it/s]

  D2_fin/C5_L2/WEAK: EX=0.031 valid=1.000 n=32 | cache=32 api=0 | 4s
Saved table: generation_results.csv, generation_results.tex


,dataset,condition,model_tier,n,EX,valid,tbl_recall,tbl_prec,col_rec_strict,col_prec_strict,col_rec_relax,col_prec_relax,ctx_tokens,cost
0,D1,C1,MID,1034,0.7573,1.0000,1.0000,0.3984,0.9217,0.1562,0.9298,0.1661,418.4816,0.0004
1,D1,C1,STRONG,1034,0.7785,0.9990,1.0000,0.3984,0.9217,0.1562,0.9298,0.1661,418.4816,0.0021
2,D1,C1,WEAK,1034,0.7466,0.9981,1.0000,0.3984,0.9217,0.1562,0.9298,0.1661,418.4816,0.0001
3,D1,C2,MID,1034,0.7447,1.0000,0.9990,0.4214,0.9035,0.2087,0.9175,0.2207,230.1847,0.0003
4,D1,C2,STRONG,1034,0.7727,0.9981,0.9990,0.4214,0.9035,0.2087,0.9175,0.2207,230.1847,0.0014
5,D1,C2,WEAK,1034,0.7379,0.9981,0.9990,0.4214,0.9035,0.2087,0.9175,0.2207,230.1847,0.0001
6,D1,C3,MID,1034,0.7215,1.0000,0.9521,0.5033,0.8743,0.1842,0.8852,0.1899,307.1103,0.0003
7,D1,C3,STRONG,1034,0.7427,0.9990,0.9521,0.5033,0.8743,0.1842,0.8852,0.1899,307.1103,0.0017
8,D1,C3,WEAK,1034,0.7060,0.9981,0.9521,0.5033,0.8743,0.1842,0.8852,0.1899,307.1103,0.0001
9,D1,C4,MID,1034,0.7553,1.0000,1.0000,0.4023,0.9153,0.1866,0.9248,0.1963,322.4749,0.0003


Saved table: generation_results_summary.csv, generation_results_summary.tex


In [3]:
# ============================================================================
# 02_run_generation.ipynb
# ----------------------------------------------------------------------------
# Beyond Full-Schema Prompting: A Graph-based Semantic Layer for Text-to-SQL
#
# Core experiment loop. For each (dataset, condition, model, question):
#   1. Load the pre-built schema context (notebook 01).
#   2. Prompt the LLM to generate SQL.
#   3. Validate with SQLGlot (Invalid SQL rate).
#   4. Execute predicted vs gold SQL on the SQLite DB; compare result sets (EX).
#   5. Compute schema-linking recall/precision vs gold tables/columns.
#   6. Record tokens, latency, estimated cost.
# Writes one tidy master table consumed by all RQ notebooks (03-08).
#
# v4 progress reporting: shows overall progress across all cells, per-cell cache
# hit/miss counts, live API-call rate and mean latency, so slowdowns are visible
# (a slow patch = real API calls; a fast patch = cache hits). SMOKE_TEST switch
# for a tiny cheap run. Anonymity: no filesystem paths are printed.
# ============================================================================


# %% [markdown]
# ## Cell 1 - Load shared configuration

# %%
%run 00_setup_and_config.ipynb


# %%
import re
import time
import sqlglot
from tqdm.auto import tqdm

print("Notebook 02 dependencies ready.")


# %% [markdown]
# ## Cell 2 - Run controls (smoke switch) + experiment plan

# %%
SMOKE_TEST = False                # flip True for a tiny cheap run
SMOKE_LIMIT = 5
FULL_LIMIT = None

GEN_MAX_TOKENS = 512

SMOKE_PLAN = [
    ("D1", ["C1", "C4"], ["STRONG"]),
]

FULL_PLAN = [
    ("D1", ["C1", "C2", "C3", "C4", "C5_L1", "C5_L2", "C5_L3"],
     ["STRONG", "MID", "WEAK"]),
    ("D2", ["C1", "C2", "C3", "C4", "C5_L1", "C5_L2", "C5_L3"],
     ["STRONG", "MID", "WEAK"]),
    ("D2_fin", ["C1", "C4", "C5_L1", "C5_L2"],
     ["STRONG", "MID", "WEAK"]),
]

if SMOKE_TEST:
    EXPERIMENT_PLAN = SMOKE_PLAN
    RUN_LIMIT = SMOKE_LIMIT
else:
    EXPERIMENT_PLAN = FULL_PLAN
    RUN_LIMIT = FULL_LIMIT

print(f"Mode: {'SMOKE' if SMOKE_TEST else 'FULL'} | "
      f"cells={sum(len(cs)*len(ts) for _, cs, ts in EXPERIMENT_PLAN)} | "
      f"limit={RUN_LIMIT}")


# %% [markdown]
# ## Cell 3 - Prompt template + output cleaning

# %%
SQL_SYSTEM = (
    "You are an expert data analyst that writes correct, executable SQLite SQL. "
    "Use ONLY the tables and columns provided in the schema context. "
    "Return a single SQL query and nothing else - no explanation, no markdown."
)

SQL_PROMPT_TEMPLATE = """{schema_context}

-- Question:
-- {question}

-- Write one SQLite query that answers the question.
-- Output only the SQL statement.
SQL:"""


def build_prompt(schema_context, question):
    return SQL_PROMPT_TEMPLATE.format(schema_context=schema_context, question=question)


def clean_sql(text):
    """Strip markdown fences / stray prose the model may add."""
    t = text.strip()
    t = re.sub(r"^```[a-zA-Z]*", "", t).strip()
    t = re.sub(r"```$", "", t).strip()
    m = re.search(r"(?is)\b(WITH|SELECT)\b", t)
    if m:
        t = t[m.start():]
    if ";" in t:
        t = t[: t.index(";") + 1]
    return t.strip()


# %% [markdown]
# ## Cell 4 - SQLGlot validation (Invalid SQL rate)

# %%
def validate_sql(sql, dialect="sqlite"):
    try:
        return (sqlglot.parse_one(sql, read=dialect) is not None), ""
    except Exception as e:
        return False, str(e)


# %% [markdown]
# ## Cell 5 - Execution-based comparison (Execution Accuracy)
# sqlite3's connect(timeout=) governs LOCK waits, not query runtime. A
# pathological SQL (cartesian blow-up) can hang forever, so we enforce a real
# wall-clock limit via set_progress_handler, which fires every N bytecode ops and
# aborts once the deadline passes.

# %%
def _db_path(dataset_key, db_id):
    return DATA_DIR / dataset_key / "db" / f"{db_id}.sqlite"


def run_sql(dataset_key, db_id, sql, wall_limit_s=10.0):
    """
    Execute SQL with a real wall-clock limit. Returns (rows, error).
    rows is a list of tuples, or None on error/timeout.
    """
    path = _db_path(dataset_key, db_id)
    if not path.exists():
        return None, "DB not found"

    con = None
    deadline = time.time() + wall_limit_s

    def _guard():
        # Called periodically by SQLite; returning non-zero aborts the query.
        return 1 if time.time() > deadline else 0

    try:
        con = sqlite3.connect(str(path), timeout=5.0)
        con.text_factory = lambda b: b.decode("utf-8", "ignore")
        con.set_progress_handler(_guard, 1_000_000)
        cur = con.cursor()
        cur.execute(sql)
        rows = cur.fetchall()
        con.close()
        return rows, ""
    except Exception as e:
        if con is not None:
            try:
                con.close()
            except Exception:
                pass
        msg = str(e)
        if "interrupted" in msg.lower() or "aborted" in msg.lower():
            return None, f"SQL wall-clock timeout (> {wall_limit_s:.0f}s)"
        return None, msg


def _has_order_by(sql):
    return bool(re.search(r"(?is)\border\s+by\b", sql))


def compare_results(pred_rows, gold_rows, ordered):
    if pred_rows is None or gold_rows is None:
        return False
    def norm(rows):
        return [tuple("" if v is None else str(v) for v in r) for r in rows]
    p, g = norm(pred_rows), norm(gold_rows)
    if ordered:
        return p == g
    from collections import Counter
    return Counter(p) == Counter(g)


def execution_match(dataset_key, db_id, pred_sql, gold_sql):
    gold_rows, gold_err = run_sql(dataset_key, db_id, gold_sql)
    pred_rows, pred_err = run_sql(dataset_key, db_id, pred_sql)
    if gold_rows is None:
        return None, pred_err, gold_err     # unscoreable (e.g. DB missing)
    return compare_results(pred_rows, gold_rows, _has_order_by(gold_sql)), pred_err, gold_err


# %% [markdown]
# ## Cell 6 - Schema-linking metrics (strict + relaxed)

# %%
def _norm(names):
    return {n.strip().lower() for n in names if n and n.strip()}

def _suffix(name):
    return name.split(".")[-1]

def link_scores_strict(selected, gold):
    g, s = _norm(gold), _norm(selected)
    if not g:
        return np.nan, np.nan
    tp = len(g & s)
    return tp / len(g), (tp / len(s) if s else 0.0)

def link_scores_relaxed(selected, gold):
    g = {_suffix(x) for x in _norm(gold)}
    s = {_suffix(x) for x in _norm(selected)}
    if not g:
        return np.nan, np.nan
    tp = len(g & s)
    return tp / len(g), (tp / len(s) if s else 0.0)

def table_scores(selected, gold):
    g, s = _norm(gold), _norm(selected)
    if not g:
        return np.nan, np.nan
    tp = len(g & s)
    return tp / len(g), (tp / len(s) if s else 0.0)


# %% [markdown]
# ## Cell 7 - Lookups for gold questions + context metadata

# %%
def load_questions(dataset_key):
    data = json.loads((DATA_DIR / dataset_key / "questions.json").read_text(encoding="utf-8"))
    return {q["qid"]: q for q in data}

def load_context_meta(dataset_key):
    path = CONTEXTS_DIR / f"{dataset_key}__context_meta.csv"
    if not path.exists():
        return None
    df = pd.read_csv(path)
    return {(r.qid, r.condition): r for r in df.itertuples()}


# %% [markdown]
# ## Cell 8 - Plan sizing (for the global progress bar)
# Pre-compute how many questions each cell has so we can show overall progress
# (cell k / K, and question n / N across the whole run).

# %%
def plan_sizes(plan, limit):
    """Return (list of (ds,cond,tier,n_questions), total_questions)."""
    cells, total = [], 0
    for ds, conds, tiers in plan:
        meta = load_context_meta(ds)
        if meta is None:
            # contexts not built; count 0 so the cell is skipped gracefully
            for cond in conds:
                for tier in tiers:
                    cells.append((ds, cond, tier, 0))
            continue
        for cond in conds:
            n = len({qid for (qid, c) in meta.keys() if c == cond})
            if limit:
                n = min(n, limit)
            for tier in tiers:
                cells.append((ds, cond, tier, n))
                total += n
    return cells, total

PLAN_CELLS, PLAN_TOTAL = plan_sizes(EXPERIMENT_PLAN, RUN_LIMIT)
print(f"Planned: {len(PLAN_CELLS)} cells, {PLAN_TOTAL} question-generations total.")


# %% [markdown]
# ## Cell 9 - Single-cell runner with cache/API-aware progress
# The progress bar postfix shows: cache-hit count, real API-call count, and the
# rolling mean latency of REAL calls only. A slow stretch with rising 'api' and
# latency means real calls; a fast stretch with rising 'cache' means cache hits.

# %%
def run_cell(dataset_key, condition, model_tier, limit=None,
             global_bar=None):
    model_name = MODEL_TIERS[model_tier]
    questions = load_questions(dataset_key)
    meta = load_context_meta(dataset_key)
    if meta is None:
        print(f"  [skip] {dataset_key}: contexts not built (run notebook 01).")
        return pd.DataFrame()

    qids = sorted({qid for (qid, cond) in meta.keys() if cond == condition})
    if limit:
        qids = qids[:limit]

    records = []
    n_cache, n_api, api_latency_sum = 0, 0, 0.0
    t_cell0 = time.time()

    bar = tqdm(qids, desc=f"{dataset_key}/{condition}/{model_tier}",
               leave=False, dynamic_ncols=True)
    for qid in bar:
        q = questions[qid]
        m = meta[(qid, condition)]
        schema_context = (PROJECT_ROOT / m.context_path).read_text(encoding="utf-8")

        resp = llm_complete(build_prompt(schema_context, q["question"]),
                            model=model_name, system=SQL_SYSTEM,
                            temperature=0.0, max_tokens=GEN_MAX_TOKENS)

        # cache/api accounting
        if resp.get("cached"):
            n_cache += 1
        else:
            n_api += 1
            api_latency_sum += resp.get("latency_s", 0.0) or 0.0
        mean_api_lat = (api_latency_sum / n_api) if n_api else 0.0
        bar.set_postfix(cache=n_cache, api=n_api,
                        api_lat=f"{mean_api_lat:.2f}s")
        if global_bar is not None:
            global_bar.update(1)

        pred_sql = clean_sql(resp["text"])
        is_valid, verr = validate_sql(pred_sql)
        if is_valid:
            match, pred_err, _ = execution_match(
                dataset_key, q["db_id"], pred_sql, q["gold_sql"])
        else:
            match, pred_err = False, verr

        sel_tables = str(m.selected_tables).split("|") if m.selected_tables else []
        sel_cols = str(m.selected_columns).split("|") if m.selected_columns else []
        t_recall, t_prec = table_scores(sel_tables, q.get("gold_tables", []))
        cs_recall, cs_prec = link_scores_strict(sel_cols, q.get("gold_columns", []))
        cr_recall, cr_prec = link_scores_relaxed(sel_cols, q.get("gold_columns", []))

        records.append({
            "dataset": dataset_key, "condition": condition,
            "model_tier": model_tier, "model_name": model_name,
            "qid": qid, "db_id": q["db_id"],
            "n_gold_tables": len(q.get("gold_tables", [])),
            "n_gold_cols": len(q.get("gold_columns", [])),
            "n_join_gold": max(0, len(q.get("gold_tables", [])) - 1),
            "exec_match": (np.nan if match is None else int(bool(match))),
            "valid_sql": int(bool(is_valid)),
            "table_recall": t_recall, "table_precision": t_prec,
            "col_recall_strict": cs_recall, "col_precision_strict": cs_prec,
            "col_recall_relaxed": cr_recall, "col_precision_relaxed": cr_prec,
            "context_tokens_est": m.context_tokens_est,
            "prompt_tokens": resp["prompt_tokens"],
            "completion_tokens": resp["completion_tokens"],
            "total_tokens": resp["total_tokens"],
            "cost_usd": estimate_cost(model_name, resp["prompt_tokens"],
                                      resp["completion_tokens"]),
            "latency_s": resp["latency_s"],
            "cached": int(resp["cached"]),
            "pred_sql": pred_sql, "pred_error": pred_err,
        })

    bar.close()
    df = pd.DataFrame(records)
    elapsed = time.time() - t_cell0
    ex = df["exec_match"].mean(skipna=True) if len(df) else np.nan
    ex_str = "n/a" if (len(df) == 0 or np.isnan(ex)) else f"{ex:.3f}"
    print(f"  {dataset_key}/{condition}/{model_tier}: "
          f"EX={ex_str} valid={df['valid_sql'].mean():.3f} n={len(df)} "
          f"| cache={n_cache} api={n_api} "
          f"| {elapsed:.0f}s")
    return df


# %% [markdown]
# ## Cell 10 - Execute the plan (with a global progress bar)
# The outer bar tracks total question-generations across ALL cells so you can see
# overall speed and ETA; each cell also prints its own cache/api breakdown.

# %%
all_results = []
done_cells = 0
global_bar = tqdm(total=PLAN_TOTAL, desc="OVERALL", position=0,
                  dynamic_ncols=True)

for dataset_key, conditions, tiers in EXPERIMENT_PLAN:
    if not (DATA_DIR / dataset_key / "questions.json").exists():
        print(f"[skip] {dataset_key}: questions.json missing.")
        continue
    for cond in conditions:
        for tier in tiers:
            done_cells += 1
            n_this = next((n for (d, c, t, n) in PLAN_CELLS
                           if d == dataset_key and c == cond and t == tier), 0)
            global_bar.set_postfix(cell=f"{done_cells}/{len(PLAN_CELLS)}",
                                   now=f"{dataset_key}/{cond}/{tier}")
            df_cell = run_cell(dataset_key, cond, tier, limit=RUN_LIMIT,
                               global_bar=global_bar)
            if len(df_cell):
                all_results.append(df_cell)

global_bar.close()
results = pd.concat(all_results, ignore_index=True) if all_results else pd.DataFrame()
if not len(results):
    print("No results produced. Check notebook 01 outputs.")


# %% [markdown]
# ## Cell 11 - Persist the master results table
# Smoke mode writes a separate file so it never overwrites a full run.

# %%
if len(results):
    out_name = "generation_results_smoke" if SMOKE_TEST else "generation_results"
    save_table(results, out_name)

    summary = (results.groupby(["dataset", "condition", "model_tier"])
               .agg(n=("qid", "count"),
                    EX=("exec_match", "mean"),
                    valid=("valid_sql", "mean"),
                    tbl_recall=("table_recall", "mean"),
                    tbl_prec=("table_precision", "mean"),
                    col_rec_strict=("col_recall_strict", "mean"),
                    col_prec_strict=("col_precision_strict", "mean"),
                    col_rec_relax=("col_recall_relaxed", "mean"),
                    col_prec_relax=("col_precision_relaxed", "mean"),
                    ctx_tokens=("context_tokens_est", "mean"),
                    cost=("cost_usd", "mean"))
               .round(4).reset_index())
    display(summary)
    save_table(summary, out_name + "_summary")

Core libraries imported.
Project directories ready.
Environment loaded. Model tiers:
  WEAK   -> gpt-4o-mini
  MID    -> gpt-5.4-mini
  STRONG -> gpt-5.4
Conditions: ['C1', 'C2', 'C3', 'C4', 'C5']
Datasets   : ['D1', 'D2', 'D2_fin', 'D3']
Grayscale plotting style configured.
Condition styles: {'C1': '0.15', 'C2': '0.3', 'C3': '0.45', 'C4': '0.6', 'C5': '0.75'}
Live check skipped (set RUN_LIVE_CHECK=True to test credentials).
Setup complete. Config snapshot saved.
Notebook 02 dependencies ready.
Mode: FULL | cells=54 | limit=None
Planned: 54 cells, 32598 question-generations total.


OVERALL:   0%|          | 0/32598 [00:00<?, ?it/s]

D1/C1/STRONG:   0%|          | 0/1034 [00:00<?, ?it/s]

  D1/C1/STRONG: EX=0.779 valid=0.999 n=1034 | cache=1034 api=0 | 23s


D1/C1/MID:   0%|          | 0/1034 [00:00<?, ?it/s]

...

  D2_fin/C5_L2/MID: EX=0.312 valid=1.000 n=32 | cache=32 api=0 | 4s


D2_fin/C5_L2/WEAK:   0%|          | 0/32 [00:00<?, ?it/s]

  D2_fin/C5_L2/WEAK: EX=0.188 valid=1.000 n=32 | cache=32 api=0 | 13s
Saved table: generation_results.csv, generation_results.tex


,dataset,condition,model_tier,n,EX,valid,tbl_recall,tbl_prec,col_rec_strict,col_prec_strict,col_rec_relax,col_prec_relax,ctx_tokens,cost
0,D1,C1,MID,1034,0.7573,1.0000,1.0000,0.3984,0.9217,0.1562,0.9298,0.1661,418.4816,0.0004
1,D1,C1,STRONG,1034,0.7785,0.9990,1.0000,0.3984,0.9217,0.1562,0.9298,0.1661,418.4816,0.0021
2,D1,C1,WEAK,1034,0.7466,0.9981,1.0000,0.3984,0.9217,0.1562,0.9298,0.1661,418.4816,0.0001
3,D1,C2,MID,1034,0.7495,1.0000,1.0000,0.4005,0.9206,0.1641,0.9287,0.1742,314.9410,0.0003
4,D1,C2,STRONG,1034,0.7785,1.0000,1.0000,0.4005,0.9206,0.1641,0.9287,0.1742,314.9410,0.0017
5,D1,C2,WEAK,1034,0.7311,0.9981,1.0000,0.4005,0.9206,0.1641,0.9287,0.1742,314.9410,0.0001
6,D1,C3,MID,1034,0.7215,1.0000,0.9521,0.5033,0.8743,0.1842,0.8852,0.1899,307.1103,0.0003
7,D1,C3,STRONG,1034,0.7427,0.9990,0.9521,0.5033,0.8743,0.1842,0.8852,0.1899,307.1103,0.0017
8,D1,C3,WEAK,1034,0.7060,0.9981,0.9521,0.5033,0.8743,0.1842,0.8852,0.1899,307.1103,0.0001
9,D1,C4,MID,1034,0.7611,1.0000,1.0000,0.3984,0.9212,0.1624,0.9292,0.1724,371.6925,0.0004


Saved table: generation_results_summary.csv, generation_results_summary.tex
